# vepyr — Build Ensembl VEP Cache

Download an Ensembl VEP offline cache and convert it to optimized Parquet files and fjall KV stores.

In [1]:
import logging

logging.basicConfig(level=logging.INFO)

In [2]:
import os

os.environ["RUST_LOG"] = "info"

In [3]:
from vepyr import build_cache

In [ ]:
!pip install ipywidgets jupyter tqdm

In [4]:
# Configure
RELEASE = 115
CACHE_DIR = "/tmp/vepyr_cache"
SPECIES = "homo_sapiens"
ASSEMBLY = "GRCh38"
CACHE_TYPE = "vep"  # "vep", "merged", or "refseq"


# Set to an existing unpacked VEP cache directory to skip download, e.g.:
# LOCAL_CACHE = "/path/to/homo_sapiens/115_GRCh38"
LOCAL_CACHE = None

In [ ]:
results = build_cache(
    release=RELEASE,
    cache_dir=CACHE_DIR,
    species=SPECIES,
    assembly=ASSEMBLY,
    cache_type=CACHE_TYPE,
    local_cache=LOCAL_CACHE,
    partitions=6,
)

INFO:datafusion_bio_function_vep.cache_builder:variation: 25 main chroms, 1879 other contigs
INFO:datafusion_bio_function_vep.cache_builder:variation: [1] 6 partitions (parallel=true)
INFO:datafusion_bio_function_vep.cache_builder:variation: [1] phase 1 done: 88.2M rows in 6 partitions
INFO:datafusion_bio_function_vep.cache_builder:variation: 1 88.2M rows
INFO:datafusion_bio_function_vep.cache_builder:variation: [2] 6 partitions (parallel=true)
INFO:datafusion_bio_function_vep.cache_builder:variation: [2] phase 1 done: 93.2M rows in 6 partitions
INFO:datafusion_bio_function_vep.cache_builder:variation: 2 93.2M rows
INFO:datafusion_bio_function_vep.cache_builder:variation: [3] 6 partitions (parallel=true)
INFO:datafusion_bio_function_vep.cache_builder:variation: [3] phase 1 done: 76.2M rows in 6 partitions
INFO:datafusion_bio_function_vep.cache_builder:variation: 3 76.2M rows
INFO:datafusion_bio_function_vep.cache_builder:variation: [4] 6 partitions (parallel=true)
INFO:datafusion_bio_f

In [ ]:
import os

for path, rows in results:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    name = os.path.basename(path)
    print(f"{name:45s} {rows:>12,} rows  {size_mb:8.1f} MB")

In [ ]:
import pyarrow.parquet as pq

# Inspect one of the generated files
variation_file = [p for p, _ in results if "variation" in p][0]
table = pq.read_table(variation_file)
print(f"Schema: {table.schema}")
print(f"Rows: {table.num_rows:,}")
table.to_pandas().head()